In [ ]:
# OLD DOWNSAMPLING 
# Rule: downsample all the cells randomly by 10%, regardless of cluster metadata  
# Problem: wipes out small clusters (immune cell types) but leaves a lot of neuronal types => bad for the scope of ARIA project 

# Settings:
DATA_DIR = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/raw"   # <-- EDIT THIS (change output folder to raw or log) 
DOWNSAMPLE_RATE = 0.10  # keep ~10%
OUTPUT_FILE = "WMB_10xv3raw_downsampled_merged.h5ad" # <-- CHANGE THIS 

# find all .h5ad files in DATA_DIR
files = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith(".h5ad")]

# print out all the files found
print(f"Found {len(files)} .h5ad files:")

# loop through each file and print them 
for f in files:
    print(" -", os.path.basename(f))

# intialize list where the downsampled cells will be stored during for loop
subset_list = []

# loop through each file, downsample cells by 10% and save downsampled cells to subset_list
for f in files:
    print(f"\nProcessing: {os.path.basename(f)}")

    # load in backed mode (does not read expression matrix into memory)
    adata = sc.read_h5ad(f, backed="r")

    # downsample cell indices
    np.random.seed(42)  # makes results reproducible

    # number of cells to keep after downsampling
    n_keep = int(adata.n_obs * DOWNSAMPLE_RATE)

    # randomly select downsampled cells based on number calculated above 
    keep_cells = np.random.choice(adata.obs_names, size=n_keep, replace=False)

    print(f"  Keeping {n_keep} of {adata.n_obs} cells ({DOWNSAMPLE_RATE*100}%)")

    # load ONLY the selected cells fully in memory
    adata_sub = sc.read_h5ad(f)[keep_cells, :].copy()

    # add selected cells to list
    subset_list.append(adata_sub)

# merge all downsampled datasets into one object 
print("\nMerging all downsampled datasets...")

merged = sc.concat(subset_list, join="outer", label="dataset", keys=[os.path.basename(f) for f in files])

print(f"Final dataset cells: {merged.n_obs:,}")
print(f"Final dataset genes: {merged.n_vars:,}")


# save downsampled file 
merged.write_h5ad(OUTPUT_FILE)

print(f"\nSaved merged downsampled dataset to: {OUTPUT_FILE}")

In [ ]:
# OLD (01/19/26)
# Rules: downsample randomly by 10% unless class has less than 1000 cells
    #if a neuronal cluster still has over 1000 cells after random downsampling, cap it at 1000 cells
# Problem: lymphoid clusters got downsampled and I ended up with less than 100 cells in those clusters 

# Settings:
DATA_DIR = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/raw" # <-- CHANGE THIS
MIN_CELLS = 1000        
DOWNSAMPLE_RATE = 0.10  
MAX_NEURONAL = 1000     # neuronal cap
OUTPUT_FILE = "20260121_WMB_10xv3_downsampled_per_class.h5ad" # <-- CHANGE THIS

# column that defines cluster/class in metadata file 
CLUSTER_COL = "cluster"
CLASS_COL = "class"



# path to metadata with cluster annotations
METADATA_FILE =  "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/metadata/WMB-10X/20241115/views/cell_metadata_with_cluster_annotation.csv"


# load metadata
cell_meta = pd.read_csv(METADATA_FILE, index_col=0)


# find all .h5ad files in DATA_DIR
files = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith(".h5ad")]


# intialize list where the downsampled cells will be stored during for loop
subset_list = []


# loop through each file, and apply rules 
for f in files:
    print(f"\nProcessing: {os.path.basename(f)}")

    # load in backed mode (does not read expression matrix into memory)
    adata = sc.read_h5ad(f, backed="r")
    
    # create new metadata columns in the 10xv3 object for cluster information
    merged_obs = adata.obs.merge(cell_meta[[CLUSTER_COL, CLASS_COL]], 
                                 left_on=adata.obs_names, 
                                 right_index=True, 
                                 how='left')

    # add metadata values in the new columns in the 10xv3 obj
    adata.obs[CLUSTER_COL] = merged_obs[CLUSTER_COL].values
    
    adata.obs[CLASS_COL] = merged_obs[CLASS_COL].values

    # intialize keep_cells list
    keep_cells = []
    
    np.random.seed(42)

    # loop over classes (highest annotation)
    for class_name, group in adata.obs.groupby(CLASS_COL):
        
        # classifying what is considered a neuronal class (if the class name contains Glut, GABA, Sero, or Dopa, it is a neuronal cluster)
        IS_NEURONAL = bool(pd.Series(class_name).str.contains(r"Glut|GABA|Dopa|Sero", regex=True, na=False).iloc[0])

        # number of cells in the class 
        n_cells = group.shape[0]
        
        # RULE: if there are <1000 cells in the class, do not downsample 
        if n_cells <= MIN_CELLS or pd.isna(class_name):
            
            # Keep all small classes (<1000 cells) (add cells to list)
            keep_cells.extend(group.index.tolist())

        # RULE: if not, downsample randomly by 10%     
        else:
            
            # downsampling
            n_keep = max(1, int(n_cells * DOWNSAMPLE_RATE))

            # number of cells to keep after downsampling
            sampled = np.random.choice(group.index, size=n_keep, replace=False)

            
            # RULE: (after downsampling) if the class is neuronal and it has >1000 cells, cap it at 1000 cells  
            if IS_NEURONAL and len(sampled) > MAX_NEURONAL:

                # cap the neuronal class
                sampled = np.random.choice(sampled, size=MAX_NEURONAL, replace=False)

                # print which neuronal class got capped
                print(f"  Neuronal cap applied to '{class_name}': capped at {MAX_NEURONAL}")

            # RULE: if not, downsample 
            else:
                print(f"  Downsampling class '{class_name}': keeping {len(sampled)} of {n_cells} cells")

            # add cells to list
            keep_cells.extend(sampled)

    # load fully in memory ONLY selected cells
    adata_sub = sc.read_h5ad(f)[keep_cells, :].copy()

    # add selected cells to object
    subset_list.append(adata_sub)

# merge all downsampled datasets into one object 
merged = sc.concat(subset_list, join="outer", label="dataset", keys=[os.path.basename(f) for f in files])

print(f"\nFinal dataset cells: {merged.n_obs:,}, genes: {merged.n_vars:,}")

# save downsampled file 
merged.write_h5ad(OUTPUT_FILE)
print(f"\nSaved merged downsampled dataset to: {OUTPUT_FILE}")

# REPEAT WITH LOG2 FILES!


In [ ]:
# OLD (01/21/26)
# Rules: 
    # 1. downsample neuronal CLASSES randomly by 10%, 
        # 1a. if there are <1000 cells in the class, don't downsample
        # 1b. after downsampling, cap neuronal classes at 1000 cells, if needed 
    # 2. downsample nonneuronal CLUSTERS randomly by 10% 
        # 2a. unless the cluster has <1000 cells

# PROBLEM: neuronal cap is added locally (within each individual file), therefore when all the data is merged together,
    # the neuronal classes still end up with >1000k cells, causing us to be stuck with way more cells (~240k cells) than we want (closer to 100k)
    # also, unannotated cells were still kept after downsampling 

# Settings:
DATA_DIR = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/raw" # <-- CHANGE THIS
MIN_CELLS = 1000        
DOWNSAMPLE_RATE = 0.10  
MAX_NEURONAL = 1000     # neuronal cap
OUTPUT_FILE = "20260121_WMB_10xv3_downsampled.h5ad" # <-- CHANGE THIS

# column that defines cluster/class in metadata
CLUSTER_COL = "cluster"
CLASS_COL = "class"



# path to metadata with cluster annotations
METADATA_FILE =  "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/metadata/WMB-10X/20241115/views/cell_metadata_with_cluster_annotation.csv"


# load metadata
cell_meta = pd.read_csv(METADATA_FILE, index_col=0)


# find all .h5ad files in DATA_DIR
files = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith(".h5ad")]


# intialize list where the downsampled cells will be stored during for loop
subset_list = []


# loop through each file, and apply rules 
for f in files:
    
    print(f"\nProcessing: {os.path.basename(f)}")
    
    adata = sc.read_h5ad(f, backed="r")
    
    # create new metadata columns in the 10xv3 object for cluster information
    merged_obs = adata.obs.merge(cell_meta[[CLUSTER_COL, CLASS_COL]], 
                                 left_on=adata.obs_names, 
                                 right_index=True, 
                                 how='left')
    
    # add metadata values in the new columns in the 10xv3 obj
    adata.obs[CLUSTER_COL] = merged_obs[CLUSTER_COL].values
    
    adata.obs[CLASS_COL] = merged_obs[CLASS_COL].values

    # intialize keep_cells list
    keep_cells = []

    np.random.seed(42)

    # loop over classes (highest annotation)
    for class_name, class_df in adata.obs.groupby(CLASS_COL):

        # if there is no class name, keep the cells and move on to next block 
            # prevents errors in rest of loop, and these unannotated cells will be removed later on 
        if pd.isna(class_name):
            
            keep_cells.extend(class_df.index.tolist())
            
            continue

        # classifying what is considered a neuronal class (if the class name has Glut, GABA, Sero, or Dopa, it is a neuronal cluster)
        IS_NEURONAL = bool(pd.Series(class_name).str.contains(r"Glut|GABA|Dopa|Sero", regex = True, na = False).iloc[0])

        
        # number of cells in the class 
        n_class = class_df.shape[0]

        # downsampling neurons by class only 
        if IS_NEURONAL:

            # RULE: if there are <1000 in the class, don't downsample 
            if n_class <= MIN_CELLS:
                
                keep_cells.extend(class_df.index.tolist())
                
                continue

            # number of cells to keep after downsampling 
            n_keep = max(1, int(n_class * DOWNSAMPLE_RATE))

            # randomly downsampling that number of cells
            sampled = np.random.choice(class_df.index, size=n_keep, replace=False)

            # RULE: if neuronal class has >1000 cells after downsampling, cap it at 1000 cells 
            if len(sampled) > MAX_NEURONAL:
                
                sampled = np.random.choice(sampled, size=MAX_NEURONAL, replace=False)
                
                print(f"  Neuronal cap: {class_name} → {MAX_NEURONAL}")

            # RULE: if not, add downsampled neuronal cells to keep_cells list
            else:
                print(f"  Neuronal downsample: {class_name} → {len(sampled)}")

            keep_cells.extend(sampled)
            
            continue

        # split by cluster for nonneuronal cell types 
        for cluster_name, cluster_df in class_df.groupby(CLUSTER_COL):

            # number of cells in cluster
            n_cluster = cluster_df.shape[0]

            # RULE: if cluster has <1000 cells, don't downsample 
            if n_cluster < MIN_CELLS:
                
                keep_cells.extend(cluster_df.index.tolist())
                
                continue

            #RULE: downsample all other nonneuronal clusters by 10%
            n_keep = max(1, int(n_cluster * DOWNSAMPLE_RATE))
            
            sampled = np.random.choice(cluster_df.index, size = n_keep, replace = False)

            # add these downsampled cells to keep_cells list
            keep_cells.extend(sampled)

    # Load fully in memory only selected cells
    adata_sub = sc.read_h5ad(f)[keep_cells, :].copy()

    # add keep_cells to subset_list after looping through each file 
    subset_list.append(adata_sub)

# merge all downsampled datasets 
merged = sc.concat(subset_list, join="outer", label="dataset", keys=[os.path.basename(f) for f in files])

print(f"\nFinal dataset cells: {merged.n_obs:,}, genes: {merged.n_vars:,}")

# save merged and downsampled file 
merged.write_h5ad(OUTPUT_FILE)

print(f"\nSaved merged downsampled dataset to: {OUTPUT_FILE}")